# Mutation Walkthrough

This notebook walks through the current symbolic-integration tree pipeline in `tree_diffusion_integration`:

`prefix expression -> AST -> canonical form -> position index -> mutation operators -> current observation state -> reverse edit target -> model tokens -> training examples -> dataloader batch`

The most important functions in that pipeline are `parse_prefix_string(...)`, `serialize_prefix_string(...)`, `canonicalize(...)`, `index_tree_positions(...)`, `local_replacement_candidates(...)`, `can_locally_replace(...)`, `local_replace_once(...)`, `sample_valid_subtree(...)`, `can_sampled_subtree_replace(...)`, `replace_subtree_by_node_id(...)`, `collect_candidate_nodes(...)`, `mutate_once(...)`, `build_observation(...)`, `first_edit_toward_target(...)`, `compute_edit_path(...)`, `TreeDiffusionTokenizer(...)`, `generate_training_example(...)`, and `make_tree_diffusion_dataloader(...)`.

The goal is to make each stage concrete: what each function takes as input, what it returns, what invariants it enforces, and why the later mutation and training stages depend on the earlier normalization steps.

The notebook is written as a runnable tutorial and intentionally stores no executed outputs.

## Mental Model

The integration task is conditional: the model sees an integrand `f` and a current candidate antiderivative `I_t`, then predicts one edit that should move `I_t` closer to a gold antiderivative `I*`.

| Symbol or term | Meaning in this notebook |
|---|---|
| `f` | The target integrand, such as `pow x INT+ 2`. |
| `I*` | A known correct antiderivative used to create supervision. It is debug metadata, not a dedicated model input field. |
| `I_t` | The current antiderivative candidate. During training it is produced by corrupting `I*`; at inference time it is whatever the search process currently has. |
| `g_t` | The derivative of the current candidate, `d/dx I_t`. If `g_t` matches `f`, the candidate differentiates correctly. |
| Forward mutation | A noising step applied to `I*` to create a plausible but usually wrong `I_t`. |
| Reverse edit | A supervised repair step from the current `I_t` toward `I*`. The label is this next edit, not the inverse of the last random mutation. |
| Observation | The inference-time state built from `(f, I_t)`: target integrand, current candidate, current derivative, and optional residual features. |
| Model input | Serialized observation fields followed by `<EDIT>`. It does not contain a separate gold-antiderivative field. |
| Model target | One edit label: `<POS_i> replacement_subtree <eos>`. |

Reading the notebook from top to bottom follows that same story. First we make trees and stable node positions, then we show how trees are corrupted, then we show how to compute the next repair edit, and finally we serialize the observation-plus-edit pair into tokens and batches.


In [23]:
from pathlib import Path
import sys
import random
from pprint import pprint
from dataclasses import asdict


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").is_dir() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not find the tree_diffusion_integration repo root from the current working directory."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Using repo root: {REPO_ROOT}")

from src.mathlang.ast import BinaryOp, Const, UnaryOp, Var
from src.mathlang.parser import parse_prefix_string
from src.mathlang.serializer import serialize_prefix_string
from src.mathlang.canonicalize import canonicalize
from src.tree_diffusion.positions import index_tree_positions
from src.tree_diffusion.mutation_grammar import (
    can_locally_replace,
    can_sampled_subtree_replace,
    local_replacement_candidates,
)
from src.tree_diffusion.mutation import (
    LOCAL_CONST_EDIT,
    LOCAL_SAME_ARITY_REPLACEMENT,
    SAMPLED_SMALL_SUBTREE_REPLACEMENT,
    collect_candidate_nodes,
    local_replace_once,
    mutate_once,
    replace_subtree_by_node_id,
    sample_valid_subtree,
)
from src.tree_diffusion.observation import DEFAULT_PROBE_POINTS, build_observation, compute_current_derivative
from src.tree_diffusion.tokenizer import TreeDiffusionTokenizer


Using repo root: /workspace/rbarket/tree_diffusion_integration


## Example Expressions And ASTs

We start with concrete prefix expressions, parse them into the math AST classes, and print both the raw dataclass representation and a small recursive tree view.

Prefix notation means every operator appears before its operands. A binary expression such as `x + sin(x)` becomes `add x sin x`; the parser knows `add` needs two child expressions and `sin` needs one. There are no parentheses in the token stream because operator arity determines where each subtree ends.

The expression vocabulary has a small number of token families:

| Token family | Examples | Meaning |
|---|---|---|
| Binary operators | `add`, `mul`, `pow`, `div` | Two-child AST nodes represented as `BinaryOp`. |
| Unary functions | `sin`, `cos`, `ln`, `exp`, `sqrt`, `tan`, ... | One-child AST nodes represented as `UnaryOp`. |
| Variable | `x` | The only variable currently accepted by the mutation grammar. |
| Named constants | `E`, `I`, `Pi` | Constants represented as `Const(symbol=...)`; `PI` and `pi` normalize to `Pi`. |
| Signed integer prefix | `INT+`, `INT-` | Start a signed integer literal. The sign token is followed by one or more digit tokens. |
| Digit tokens | `0` through `9` | Digits inside integer literals, so `INT- 1 2` means `-12`. |

`parse_prefix_string(...)` is the entry point from notebook-friendly text into the typed AST used everywhere else in the mutation code. It tokenizes on whitespace, recognizes `INT+` / `INT-` followed by digit tokens as numeric constants, parses unary operators into `UnaryOp`, and parses every binary operator, including `add` and `mul`, into `BinaryOp`.

`serialize_prefix_string(...)` is the inverse view we use throughout the walkthrough. It turns the AST back into the prefix token stream that the notebook prints in before / after comparisons, so it is the easiest way to see how a mutation changed structure. Numeric constants round-trip through the serializer's token format, including fractions, and a parsed constant `div` can already collapse to a `Const` when both sides are numeric and the denominator is nonzero.

The examples below intentionally include simple functions, multi-token integers, named constants, and nested prefix expressions so the AST shape is visible before we start normalizing and mutating trees.


In [24]:
EXAMPLE_EXPRESSIONS = [
    "sin x",
    "pow x INT+ 2",
    "INT- 1 2",
    "mul Pi x",
    "mul x mul INT+ 1 INT+ 2",
    "add sin x pow x INT+ 2",
    "add mul INT- 1 x exp add x INT+ 1",
]


def expr_tree_lines(node, indent: str = "") -> list[str]:
    if isinstance(node, Const):
        if node.is_named:
            return [f"{indent}Const(symbol={node.symbol!r})"]
        return [f"{indent}Const(value={node.value})"]

    if isinstance(node, Var):
        return [f"{indent}Var(name={node.name!r})"]

    if isinstance(node, UnaryOp):
        lines = [f"{indent}UnaryOp(op={node.op!r})", f"{indent}  operand:"]
        lines.extend(expr_tree_lines(node.operand, indent + "    "))
        return lines

    if isinstance(node, BinaryOp):
        lines = [f"{indent}BinaryOp(op={node.op!r})", f"{indent}  left:"]
        lines.extend(expr_tree_lines(node.left, indent + "    "))
        lines.append(f"{indent}  right:")
        lines.extend(expr_tree_lines(node.right, indent + "    "))
        return lines

    raise TypeError(f"Unsupported node type: {type(node)!r}")


for expression in EXAMPLE_EXPRESSIONS:
    parsed = parse_prefix_string(expression)
    print("=" * 80)
    print(f"Input prefix:            {expression}")
    print(f"Parsed dataclass repr:   {parsed!r}")
    print(f"Serialized prefix:       {serialize_prefix_string(parsed)}")
    print("Tree view:")
    print("\n".join(expr_tree_lines(parsed)))


Input prefix:            sin x
Parsed dataclass repr:   UnaryOp(token_start=0, token_end=2, op='sin', operand=Var(token_start=1, token_end=2, name='x'))
Serialized prefix:       sin x
Tree view:
UnaryOp(op='sin')
  operand:
    Var(name='x')
Input prefix:            pow x INT+ 2
Parsed dataclass repr:   BinaryOp(token_start=0, token_end=4, op='pow', left=Var(token_start=1, token_end=2, name='x'), right=Const(token_start=2, token_end=4, value=Fraction(2, 1), symbol=None))
Serialized prefix:       pow x INT+ 2
Tree view:
BinaryOp(op='pow')
  left:
    Var(name='x')
  right:
    Const(value=2)
Input prefix:            INT- 1 2
Parsed dataclass repr:   Const(token_start=0, token_end=3, value=Fraction(-12, 1), symbol=None)
Serialized prefix:       INT- 1 2
Tree view:
Const(value=-12)
Input prefix:            mul Pi x
Parsed dataclass repr:   BinaryOp(token_start=0, token_end=3, op='mul', left=Const(token_start=1, token_end=2, value=None, symbol='Pi'), right=Var(token_start=2, token_end=3, n

## Canonicalization And Position Indexing

`canonicalize(...)` is the first normalization pass applied before mutation. It does not try to prove arbitrary algebraic equivalence; it rewrites the AST into one consistent structural form that the mutation code can index and compare reliably.

Concretely, `canonicalize(...)`:

- normalizes operator and named-constant tokens,
- recursively canonicalizes children first,
- flattens same-operator binary `add` / `mul` chains into a temporary term list,
- sorts commutative associative terms by a structural key and rebuilds a right-nested `BinaryOp` chain,
- folds numeric `div(const, const)` into a single `Const` when the denominator is nonzero,
- strips top-level additive constants that do not contain `x`, so `add INT+ 7 add x INT+ 2` becomes just `x`. Note that this should only be done to integrals, not integrands

For `add` and `mul`, flattening means canonicalization temporarily treats nested same-operator trees as one list of terms, so `add add a b c` and `add a add b c` both become the term list `[a, b, c]` before sorting and rebuilding the standard right-nested tree.

This means canonicalization is a structural cleanup pass, not a full computer algebra system. It makes equivalent local shapes line up when they differ only by ordering, aliases, or simple constant-only fragments, but it does not try to solve arbitrary identities.

Mutation always starts from the canonical tree, not the raw parsed tree. That matters because node ids, token spans, subtree sizes, and equality checks are all defined on this normalized form.

`index_tree_positions(...)` then walks the canonical tree in preorder and records one `NodePosition` per node. Each row tells us:

- `node_id`: the preorder id later used by subtree replacement and `<POS_i>` labels,
- `token_start` / `token_end`: the half-open span in the serialized canonical prefix token stream,
- `depth`: root depth is 0,
- `parent_id`: the preorder id of the parent node,
- `production_family`: a coarse grammar family such as `CONST`, `VAR`, `UNARY_EXPR`, `ADD_EXPR`, `MUL_EXPR`, `POW_EXPR`, or `DIV_EXPR`,
- `subtree_size`: the number of operator nodes in that subtree; leaves have size 0,
- `is_mutable`: whether the node has a legal mutation under the optional `sigma_small` size cap.

Half-open spans follow the usual Python convention: a node with `token_start=3` and `token_end=7` covers `serialized_tokens[3:7]`. Preorder means the root is visited first, then the left/only child subtree, then the right child subtree for binary nodes.


In [25]:
def print_table(rows: list[dict], columns: list[str]) -> None:
    if not rows:
        print("<no rows>")
        return

    widths = {
        column: max(len(column), max(len(str(row[column])) for row in rows))
        for column in columns
    }
    header = " | ".join(f"{column:<{widths[column]}}" for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(f"{str(row[column]):<{widths[column]}}" for column in columns))


def show_canonicalization(expression: str) -> None:
    original = parse_prefix_string(expression)
    canonical = canonicalize(original)
    print("-" * 80)
    print(f"Original:  {expression}")
    print(f"Canonical: {serialize_prefix_string(canonical)}")


for expression in EXAMPLE_EXPRESSIONS:
    show_canonicalization(expression)

CANONICALIZATION_FOCUS_EXAMPLES = [
    ("commutative sorting", "add pow x INT+ 2 sin x"),
    ("add flattening", "add add sin x pow x INT+ 2 x"),
    ("mul flattening", "mul x mul INT+ 1 INT+ 2"),
    ("numeric division folding", "div INT+ 6 INT+ 3"),
    ("top-level additive constant stripping", "add INT+ 7 add x INT+ 2"),
    ("named constant alias normalization", "mul pi x"),
]

print("-" * 80)
print("Focused canonicalization examples:")
canonical_rows = []
for label, expression in CANONICALIZATION_FOCUS_EXAMPLES:
    canonical_rows.append(
        {
            "case": label,
            "input": expression,
            "canonical": serialize_prefix_string(canonicalize(parse_prefix_string(expression))),
        }
    )
print_table(canonical_rows, ["case", "input", "canonical"])

print("\nPosition index for 'add sin x pow x INT+ 2' with sigma_small=2:")
index_expr = canonicalize(parse_prefix_string("add sin x pow x INT+ 2"))
index = index_tree_positions(index_expr, sigma_small=2)
print(f"Serialized tokens: {tuple(serialize_prefix_string(index_expr).split())}")

rows = []
for position in index.positions:
    row = asdict(position)
    row["subtree"] = serialize_prefix_string(index.node_id_to_node[position.node_id])
    rows.append(row)

print_table(
    rows,
    [
        "node_id",
        "parent_id",
        "depth",
        "production_family",
        "op",
        "token_start",
        "token_end",
        "subtree_size",
        "is_mutable",
        "subtree",
    ],
)

print("\nHow to read one span:")
example_position = index.positions[3]
covered_tokens = index.serialized_tokens[example_position.token_start : example_position.token_end]
print(
    f"node_id={example_position.node_id} covers tokens "
    f"[{example_position.token_start}:{example_position.token_end}] -> {covered_tokens}"
)
print(
    "With sigma_small=2, nodes whose subtree_size is greater than 2 are shown "
    "but are not selected as mutable candidates."
)


--------------------------------------------------------------------------------
Original:  sin x
Canonical: sin x
--------------------------------------------------------------------------------
Original:  pow x INT+ 2
Canonical: pow x INT+ 2
--------------------------------------------------------------------------------
Original:  INT- 1 2
Canonical: INT- 1 2
--------------------------------------------------------------------------------
Original:  mul Pi x
Canonical: mul Pi x
--------------------------------------------------------------------------------
Original:  mul x mul INT+ 1 INT+ 2
Canonical: mul INT+ 1 mul INT+ 2 x
--------------------------------------------------------------------------------
Original:  add sin x pow x INT+ 2
Canonical: add sin x pow x INT+ 2
--------------------------------------------------------------------------------
Original:  add mul INT- 1 x exp add x INT+ 1
Canonical: add exp add INT+ 1 x mul INT- 1 x
-------------------------------------------

## Mutation Decision Map

The mutation engine has three current edit families. They all replace one selected node in the canonical tree, but they differ in how much of the selected subtree they are allowed to change.

| Mutation kind | Applies to | What may change | What must stay fixed | Why it exists |
|---|---|---|---|---|
| `local_const_edit` | Numeric or named `Const` leaves | The constant value or symbol | The selected node stays a constant leaf | Gives constants a direct small-step path such as `5 -> 3` or `Pi -> E`. |
| `local_same_arity_replacement` | Leaves, unary nodes, and binary nodes with legal alternatives | The leaf kind/value, or only the operator label at the selected root | Operator children and arity stay unchanged | Makes conservative edits like `sin x -> cos x` or `pow x INT+ 2 -> div x INT+ 2`. |
| `sampled_small_subtree_replacement` | Any supported expression node selected under `sigma_small` | The whole selected subtree | Only the replacement must be a supported small expression | Allows structural jumps like `x -> add x INT+ 1` or `pow x INT+ 2 -> sin x`. |

The important distinction is shape preservation. Local replacement is deliberately narrow: it preserves leaf/unary/binary shape and, for operators, preserves the existing children. Sampled subtree replacement is deliberately broad: it samples from the full `EXPR` grammar, so the replacement root can be a different kind of node from the original.

`collect_candidate_nodes(...)` chooses which existing nodes may be selected. `sigma_small` limits that selection by `subtree_size`, and the sampled-subtree branch also uses `sigma_small` as the size budget for the replacement proposal.


## Local Same-Shape Replacement

Local replacement is the conservative mutation path. It changes the selected node in place without resampling its whole interior.

`local_replacement_candidates(node)` returns abstract replacement specs, not concrete AST nodes. For leaves, the spec says which leaf kind is allowed (`numeric_const`, `named_const`, or `var`). For operators, the spec fixes the replacement shape, operator label, and child count. A later step materializes one of those specs into an actual replacement expression.

`can_locally_replace(source, candidate)` enforces the exact local invariants:

- `Leaf <-> Leaf`
- `UnaryOp <-> UnaryOp`
- `BinaryOp <-> BinaryOp`

For operator nodes, the children must stay exactly the same and only the root label changes. For leaf nodes, the replacement must still be a leaf, must differ from the original node, and `Var` is limited to `x`. Since `add` and `mul` are ordinary binary operators now, they can participate in binary local replacements when the two children are unchanged.

The key mental shortcut: if changing the selected node would require adding, removing, or editing one of its children, it is not local replacement. That broader move belongs to sampled subtree replacement.

`local_replace_once(expr, selected_node_id, rng)` canonicalizes the input, reindexes it, looks up the selected current-tree node, chooses a legal local replacement candidate, materializes it, replaces the subtree by node id, and canonicalizes the final result.


In [26]:
def format_spec(spec) -> str:
    if spec.leaf_kind is not None:
        return f"leaf:{spec.leaf_kind}"
    return f"{spec.shape}:{spec.op} (child_count={spec.child_count})"


def show_local_candidates(label: str, node) -> None:
    print("-" * 80)
    print(label)
    print(f"Node: {serialize_prefix_string(node)}")
    print([format_spec(spec) for spec in local_replacement_candidates(node)])


const_leaf = parse_prefix_string("INT+ 2")
var_leaf = parse_prefix_string("x")
unary_expr = parse_prefix_string("sin x")
pow_expr = parse_prefix_string("pow x INT+ 2")
add_expr = parse_prefix_string("add x pow x INT+ 2")

show_local_candidates("Leaf constant candidates", const_leaf)
show_local_candidates("Variable x candidates", var_leaf)
show_local_candidates("Unary candidates", unary_expr)
show_local_candidates("Binary pow candidates", pow_expr)
show_local_candidates("Binary add candidates", add_expr)

print()
print("Selected can_locally_replace(...) checks:")
checks = [
    (
        "legal unary: sin(x) -> cos(x)",
        parse_prefix_string("sin x"),
        parse_prefix_string("cos x"),
    ),
    (
        "legal binary: pow(x, 2) -> div(x, 2)",
        parse_prefix_string("pow x INT+ 2"),
        parse_prefix_string("div x INT+ 2"),
    ),
    (
        "legal binary add->mul with same children",
        parse_prefix_string("add x pow x INT+ 2"),
        parse_prefix_string("mul x pow x INT+ 2"),
    ),
    (
        "illegal binary replacement with changed child",
        parse_prefix_string("add x pow x INT+ 2"),
        parse_prefix_string("mul x INT+ 2"),
    ),
]

for label, source, target in checks:
    print("-" * 80)
    print(label)
    print(f"source: {serialize_prefix_string(source)}")
    print(f"target: {serialize_prefix_string(target)}")
    print(f"can_locally_replace: {can_locally_replace(source, target)}")


--------------------------------------------------------------------------------
Leaf constant candidates
Node: INT+ 2
['leaf:numeric_const', 'leaf:named_const', 'leaf:var']
--------------------------------------------------------------------------------
Variable x candidates
Node: x
['leaf:numeric_const', 'leaf:named_const']
--------------------------------------------------------------------------------
Unary candidates
Node: sin x
['unary:ln (child_count=1)', 'unary:exp (child_count=1)', 'unary:sqrt (child_count=1)', 'unary:abs (child_count=1)', 'unary:sign (child_count=1)', 'unary:cos (child_count=1)', 'unary:tan (child_count=1)', 'unary:cot (child_count=1)', 'unary:sec (child_count=1)', 'unary:csc (child_count=1)', 'unary:sinh (child_count=1)', 'unary:cosh (child_count=1)', 'unary:tanh (child_count=1)', 'unary:coth (child_count=1)', 'unary:sech (child_count=1)', 'unary:csch (child_count=1)', 'unary:asin (child_count=1)', 'unary:acos (child_count=1)', 'unary:atan (child_count=1)', 

In [27]:
def show_local_mutation(label: str, expr, selected_node_id: int, seed: int) -> None:
    result = local_replace_once(expr, selected_node_id=selected_node_id, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"local_replace_once returned None for {label}")

    print("-" * 80)
    print(label)
    print(f"Canonical input expression: {serialize_prefix_string(canonicalize(expr))}")
    print(f"Selected node id:          {selected_node_id}")
    print(f"Original subtree:          {serialize_prefix_string(result.original_subtree)}")
    print(f"Replacement subtree:       {serialize_prefix_string(result.replacement_subtree)}")
    print(f"Final mutated expression:  {serialize_prefix_string(result.mutated_expr)}")


show_local_mutation(
    "Leaf replacement (constant leaf)",
    parse_prefix_string("pow x INT+ 5"),
    selected_node_id=2,
    seed=1,
)
show_local_mutation(
    "Unary replacement",
    parse_prefix_string("sin x"),
    selected_node_id=0,
    seed=1,
)
show_local_mutation(
    "Binary pow replacement",
    parse_prefix_string("pow x INT+ 2"),
    selected_node_id=0,
    seed=0,
)
show_local_mutation(
    "Binary add/mul replacement",
    parse_prefix_string("add x pow x INT+ 2"),
    selected_node_id=0,
    seed=0,
)


--------------------------------------------------------------------------------
Leaf replacement (constant leaf)
Canonical input expression: pow x INT+ 5
Selected node id:          2
Original subtree:          INT+ 5
Replacement subtree:       INT+ 6
Final mutated expression:  pow x INT+ 6
--------------------------------------------------------------------------------
Unary replacement
Canonical input expression: sin x
Selected node id:          0
Original subtree:          sin x
Replacement subtree:       sign x
Final mutated expression:  sign x
--------------------------------------------------------------------------------
Binary pow replacement
Canonical input expression: pow x INT+ 2
Selected node id:          0
Original subtree:          pow x INT+ 2
Replacement subtree:       mul x INT+ 2
Final mutated expression:  mul INT+ 2 x
--------------------------------------------------------------------------------
Binary add/mul replacement
Canonical input expression: add x pow x INT

## Sampled Small Subtree Replacement

Sampled subtree replacement is the broader structural path. Instead of only changing the root label of the selected node, it can resample an entirely new small `Expr` subtree and then splice that subtree into the canonical expression.

`sample_valid_subtree(family, sigma_small, rng)` is the constructor for these proposals. The important change in the current implementation is that the sampled-subtree mutation path now calls `sample_valid_subtree("EXPR", sigma_small, rng)`, so the replacement can have a different root kind from the selected subtree.

`can_sampled_subtree_replace(source, candidate)` is now intentionally broad: it only checks that both trees are supported current-grammar expressions. That means cross-shape jumps such as `pow(x, 2) -> sin(x)`, `sin(x) -> div(x, 2)`, `x -> add(x, 1)`, and `Const(2) -> exp(x)` are legal here even though they are all illegal under local replacement.

`replace_subtree_by_node_id(expr, node_id, replacement)` does the actual tree surgery. It walks the tree using the same preorder numbering scheme as `index_tree_positions(...)` and swaps in the replacement when it reaches the requested node id. By itself, this function does not check whether the replacement is legal and it does not canonicalize the result; the caller is responsible for both of those steps.

The next cell shows sampled proposals and three increasingly rich subtree-replacement patterns:

- a root cross-shape jump,
- a leaf-to-nonleaf jump,
- and a larger non-root subtree replacement inside a nested expression.

When reading the output, compare `Local replacement legal` with `Sampled subtree replacement`. The first answers whether the edit preserves shape and children; the second answers whether a broad small-subtree splice is allowed.


In [28]:
def first_node_id(expr, predicate) -> int:
    index = index_tree_positions(canonicalize(expr))
    for position in index.positions:
        node = index.node_id_to_node[position.node_id]
        if predicate(node):
            return position.node_id
    raise LookupError("No matching node found.")


def operator_count(node) -> int:
    if isinstance(node, (Const, Var)):
        return 0
    return 1 + sum(operator_count(child) for child in node.children())


def sampled_expr_rows(sigma_small: int, seeds: range) -> list[dict]:
    rows = []
    for seed in seeds:
        proposal = sample_valid_subtree("EXPR", sigma_small=sigma_small, rng=random.Random(seed))
        rows.append(
            {
                "seed": seed,
                "proposal": serialize_prefix_string(proposal),
                "root_type": type(proposal).__name__,
                "subtree_size": operator_count(proposal),
            }
        )
    return rows


def show_subtree_replacement_case(
    label: str,
    expression: str,
    selected_node_id: int,
    replacement_expression: str,
) -> None:
    canonical_expr = canonicalize(parse_prefix_string(expression))
    replacement = parse_prefix_string(replacement_expression)
    index = index_tree_positions(canonical_expr)
    original_subtree = index.node_id_to_node[selected_node_id]
    mutated_raw = replace_subtree_by_node_id(canonical_expr, selected_node_id, replacement)
    mutated = canonicalize(mutated_raw)

    print("-" * 80)
    print(label)
    print(f"Canonical input:                {serialize_prefix_string(canonical_expr)}")
    print(f"Selected node id:               {selected_node_id}")
    print(f"Original selected subtree:      {serialize_prefix_string(original_subtree)}")
    print(f"Replacement subtree:            {serialize_prefix_string(replacement)}")
    print(f"Local replacement legal:        {can_locally_replace(original_subtree, replacement)}")
    print(f"Sampled subtree replacement:    {can_sampled_subtree_replace(original_subtree, replacement)}")
    print(f"Canonicalized final expression: {serialize_prefix_string(mutated)}")


print("A few sampled EXPR proposals with sigma_small=2:")
print_table(sampled_expr_rows(sigma_small=2, seeds=range(6)), ["seed", "proposal", "root_type", "subtree_size"])

show_subtree_replacement_case(
    "Root cross-shape subtree jump",
    "pow x INT+ 2",
    selected_node_id=0,
    replacement_expression="sin x",
)

show_subtree_replacement_case(
    "Leaf to non-leaf subtree jump",
    "x",
    selected_node_id=0,
    replacement_expression="mul x INT+ 2",
)

complex_expression = "add div sin x INT+ 2 add mul pow x INT+ 3 cos x ln x"
complex_node_id = first_node_id(
    parse_prefix_string(complex_expression),
    predicate=lambda node: isinstance(node, BinaryOp) and node.op == "div",
)
show_subtree_replacement_case(
    "Nested cross-shape replacement inside a larger tree",
    complex_expression,
    selected_node_id=complex_node_id,
    replacement_expression="mul add x INT+ 1 exp x",
)


A few sampled EXPR proposals with sigma_small=2:
seed | proposal                | root_type | subtree_size
-----+-------------------------+-----------+-------------
0    | pow div INT+ 1 INT+ 3 I | BinaryOp  | 1           
1    | INT+ 3                  | Const     | 0           
2    | div INT+ 1 add x INT+ 3 | BinaryOp  | 2           
3    | div INT+ 1 INT+ 3       | Const     | 0           
4    | div INT+ 1 INT+ 2       | Const     | 0           
5    | atanh INT+ 3            | UnaryOp   | 1           
--------------------------------------------------------------------------------
Root cross-shape subtree jump
Canonical input:                pow x INT+ 2
Selected node id:               0
Original selected subtree:      pow x INT+ 2
Replacement subtree:            sin x
Local replacement legal:        False
Sampled subtree replacement:    True
Canonicalized final expression: sin x
--------------------------------------------------------------------------------
Leaf to non-leaf sub

## Full Engine Walkthrough

`mutate_once(...)` is the full forward noising engine. It starts from a good or current expression, picks one mutable location, applies one legal mutation, and returns the canonicalized mutated expression plus metadata about what changed.

In the current implementation it performs this pipeline:

1. canonicalize the input expression,
2. index canonical positions with `index_tree_positions(...)`,
3. group mutable nodes by production family with `collect_candidate_nodes(...)`,
4. choose one family and one node from that family,
5. choose one mutation path for that node: `local_const_edit`, `local_same_arity_replacement`, or `sampled_small_subtree_replacement`,
6. build a concrete replacement subtree,
7. apply it with `replace_subtree_by_node_id(...)`,
8. canonicalize the mutated tree and reject no-op results.

`collect_candidate_nodes(...)` is the bridge between indexing and mutation selection: it runs canonicalization plus indexing and returns the mutable `NodePosition` records grouped by production family, which is exactly the pool `mutate_once(...)` samples from internally.

Two private helpers are worth knowing about when reading the source. `_sample_mutation_kind(...)` decides which of the three mutation paths are legal for the selected node, and `_apply_replacement(...)` performs the replacement plus the final canonicalization and no-op check. The subtree branch now always samples from the broad `EXPR` family, so the returned replacement can have a different root type from the selected subtree.

The returned `MutationResult` records both what changed and where it changed in the pre-mutation canonical tree:

- `selected_node_id`: preorder id of the mutated node,
- `selected_family`: production family that was sampled,
- `mutation_kind`: the actual mutation path that produced the result,
- `selected_token_start` / `selected_token_end`: half-open token span in the canonical input serialization,
- `original_subtree`: the canonical subtree that was selected,
- `replacement_subtree`: the raw subtree proposal inserted before final canonicalization,
- `mutated_expr`: the final canonicalized expression returned to the caller.

Because of the final canonicalization step, `mutated_expr` can look more simplified or reordered than the raw `replacement_subtree` might suggest. That is expected: the mutation proposes a structural edit, then canonicalization restores the tree to the normalized representation used everywhere else.


In [29]:
def summarize_candidate_pool(expression: str, sigma_small: int) -> dict[str, list[dict]]:
    canonical_expr = canonicalize(parse_prefix_string(expression))
    index = index_tree_positions(canonical_expr, sigma_small=sigma_small)
    candidates = collect_candidate_nodes(canonical_expr, sigma_small=sigma_small)
    return {
        family: [
            {
                "node_id": position.node_id,
                "span": (position.token_start, position.token_end),
                "subtree_size": position.subtree_size,
                "subtree": serialize_prefix_string(index.node_id_to_node[position.node_id]),
            }
            for position in positions
        ]
        for family, positions in sorted(candidates.items())
    }


def summarize_mutation_result(result) -> dict:
    return {
        "selected_node_id": result.selected_node_id,
        "selected_family": result.selected_family,
        "mutation_kind": result.mutation_kind,
        "selected_token_start": result.selected_token_start,
        "selected_token_end": result.selected_token_end,
        "original_subtree": serialize_prefix_string(result.original_subtree),
        "replacement_subtree": serialize_prefix_string(result.replacement_subtree),
        "mutated_expr": serialize_prefix_string(result.mutated_expr),
    }


def run_mutate_once(expression: str, sigma_small: int, seed: int) -> None:
    candidate_pool = summarize_candidate_pool(expression, sigma_small)
    result = mutate_once(parse_prefix_string(expression), sigma_small=sigma_small, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"mutate_once returned None for {expression}")

    print("-" * 80)
    print(f"Input expression: {expression}")
    print(f"sigma_small:      {sigma_small}")
    print(f"seed:             {seed}")
    print("Candidate pool by family:")
    pprint(candidate_pool)
    print("MutationResult:")
    pprint(summarize_mutation_result(result))


run_mutate_once("pow x INT+ 2", sigma_small=2, seed=0)
run_mutate_once("add sin x pow x INT+ 2", sigma_small=2, seed=0)
run_mutate_once("add div sin x INT+ 2 add mul pow x INT+ 3 cos x ln x", sigma_small=3, seed=4)


--------------------------------------------------------------------------------
Input expression: pow x INT+ 2
sigma_small:      2
seed:             0
Candidate pool by family:
{'CONST': [{'node_id': 2,
            'span': (2, 4),
            'subtree': 'INT+ 2',
            'subtree_size': 0}],
 'POW_EXPR': [{'node_id': 0,
               'span': (0, 4),
               'subtree': 'pow x INT+ 2',
               'subtree_size': 1}],
 'VAR': [{'node_id': 1, 'span': (1, 2), 'subtree': 'x', 'subtree_size': 0}]}
MutationResult:
{'mutated_expr': 'mul INT+ 2 x',
 'mutation_kind': 'local_same_arity_replacement',
 'original_subtree': 'pow x INT+ 2',
 'replacement_subtree': 'mul x INT+ 2',
 'selected_family': 'POW_EXPR',
 'selected_node_id': 0,
 'selected_token_end': 4,
 'selected_token_start': 0}
--------------------------------------------------------------------------------
Input expression: add sin x pow x INT+ 2
sigma_small:      2
seed:             0
Candidate pool by family:
{'CONST': [{'

## Mutation Kind Coverage

The earlier sections show the mechanics of local replacement and sampled subtree replacement. The next cell pins one deterministic `mutate_once(...)` example for each current mutation kind emitted by the engine:

- `local_const_edit`
- `local_same_arity_replacement`
- `sampled_small_subtree_replacement`

`MutationResult` now carries an explicit `mutation_kind` field, so the notebook can show exactly which path fired instead of inferring it afterwards. The subtree demo below intentionally looks for a cross-shape sampled subtree replacement rather than a same-shape one.


In [30]:
def find_mutation_by_kind(
    expression: str,
    sigma_small: int,
    expected_kind: str,
    *,
    require_cross_shape_subtree: bool = False,
    predicate=None,
    max_seed: int = 256,
):
    parsed = parse_prefix_string(expression)
    for seed in range(max_seed):
        result = mutate_once(parsed, sigma_small=sigma_small, rng=random.Random(seed))
        if result is None or result.mutation_kind != expected_kind:
            continue
        if require_cross_shape_subtree:
            same_type = type(result.original_subtree) is type(result.replacement_subtree)
            if same_type:
                continue
        if predicate is not None and not predicate(result):
            continue
        return seed, result
    raise RuntimeError(
        f"Could not find a mutation of kind {expected_kind!r} for {expression!r} with sigma_small={sigma_small}."
    )


MUTATION_KIND_DEMOS = [
    {
        "mutation_kind": LOCAL_CONST_EDIT,
        "expression": "pow x INT+ 5",
        "sigma_small": 0,
        "require_cross_shape_subtree": False,
        "predicate": lambda result: isinstance(result.original_subtree, Const),
    },
    {
        "mutation_kind": LOCAL_SAME_ARITY_REPLACEMENT,
        "expression": "sin x",
        "sigma_small": 1,
        "require_cross_shape_subtree": False,
        "predicate": lambda result: isinstance(result.original_subtree, UnaryOp),
    },
    {
        "mutation_kind": SAMPLED_SMALL_SUBTREE_REPLACEMENT,
        "expression": "pow x INT+ 2",
        "sigma_small": 2,
        "require_cross_shape_subtree": True,
        "predicate": None,
    },
]

mutation_rows = []
for demo in MUTATION_KIND_DEMOS:
    seed, result = find_mutation_by_kind(
        demo["expression"],
        demo["sigma_small"],
        demo["mutation_kind"],
        require_cross_shape_subtree=demo["require_cross_shape_subtree"],
        predicate=demo["predicate"],
    )

    mutation_rows.append(
        {
            "mutation_kind": result.mutation_kind,
            "expression": demo["expression"],
            "sigma_small": demo["sigma_small"],
            "seed": seed,
            "selected_node_id": result.selected_node_id,
            "selected_family": result.selected_family,
            "original_subtree": serialize_prefix_string(result.original_subtree),
            "replacement_subtree": serialize_prefix_string(result.replacement_subtree),
            "mutated_expr": serialize_prefix_string(result.mutated_expr),
        }
    )

print("One deterministic mutate_once(...) example for each current mutation kind:")
print_table(
    mutation_rows,
    [
        "mutation_kind",
        "expression",
        "sigma_small",
        "seed",
        "selected_node_id",
        "selected_family",
        "original_subtree",
        "replacement_subtree",
        "mutated_expr",
    ],
)


One deterministic mutate_once(...) example for each current mutation kind:
mutation_kind                     | expression   | sigma_small | seed | selected_node_id | selected_family | original_subtree | replacement_subtree                | mutated_expr                                 
----------------------------------+--------------+-------------+------+------------------+-----------------+------------------+------------------------------------+----------------------------------------------
local_const_edit                  | pow x INT+ 5 | 0           | 2    | 2                | CONST           | INT+ 5           | INT+ 1                             | pow x INT+ 1                                 
local_same_arity_replacement      | sin x        | 1           | 2    | 0                | UNARY_EXPR      | sin x            | cosh x                             | cosh x                                       
sampled_small_subtree_replacement | pow x INT+ 2 | 2           | 5    | 1        

## Observation State

Phase 3 adds the current observation object that will later become the model input. Given a target integrand `f` and a current antiderivative candidate `I_t`, `build_observation(...)` packages the current canonical antiderivative, its current derivative `g_t = d/dx I_t`, an optional symbolic residual `simplify(g_t - f)`, and optional numeric probe features.

This observation is intentionally inference-time only. It includes the target integrand `f` because symbolic integration is a conditional task, but it does **not** include the reverse edit path or the gold target antiderivative label `I*`. Those remain separate supervision/debug fields for training.

The residual features answer the question, "how does the derivative of my current candidate differ from the requested integrand?"

| Observation field | Source | Purpose |
|---|---|---|
| `target_integrand` | The requested `f` | Conditions the model on the problem to solve. |
| `current_antiderivative` | The current `I_t` | Tells the model what tree it is editing. |
| `current_derivative` | Differentiating `I_t` | Gives an explicit view of what the current candidate does. |
| `symbolic_residual` | `current_derivative - target_integrand` | Shows the symbolic error when conversion/simplification succeeds. |
| `numeric_probes` | Residual evaluated at fixed probe points | Provides coarse numeric evidence, including nonfinite/complex warnings. |
| `status` and `warnings` | Build-time diagnostics | Let generation and training code inspect partial failures. |

The next cells show both a clean observation, where `I_t` already differentiates back to `f`, and a corrupted observation produced by a forward mutation.


In [31]:
OBS_TARGET_INTEGRAND = "add pow x INT+ 2 cos x"
OBS_GOLD_ANTIDERIVATIVE = "add div pow x INT+ 3 INT+ 3 sin x"
OBS_SIGMA_SMALL = 2
OBS_MUTATION_SEED = 0


def as_prefix(expr) -> str | None:
    return None if expr is None else serialize_prefix_string(expr)


def show_observation(label: str, observation) -> None:
    print("-" * 80)
    print(label)
    print(f"target integrand:       {as_prefix(observation.target_integrand)}")
    print(f"current antiderivative: {as_prefix(observation.current_antiderivative)}")
    print(f"current derivative:     {as_prefix(observation.current_derivative)}")
    print(f"symbolic residual:      {as_prefix(observation.symbolic_residual)}")
    print(f"residual_mode:          {observation.residual_mode}")
    print(f"status:                 {observation.status}")
    print(f"warnings:               {list(observation.warnings)}")
    if observation.numeric_probes is None:
        print("numeric probes:         <disabled>")
        return

    probes = observation.numeric_probes
    print(f"probe points:           {probes.probe_points}")
    print(f"mean_abs_residual:      {probes.mean_abs_residual}")
    print(f"mean_squared_residual:  {probes.mean_squared_residual}")
    print(f"max_abs_residual:       {probes.max_abs_residual}")
    print(f"fraction_finite:        {probes.fraction_finite}")

    probe_rows = [
        {"x": point, "residual": value, "finite": finite}
        for point, value, finite in zip(
            probes.probe_points,
            probes.residual_values,
            probes.finite_mask,
        )
    ]
    print_table(probe_rows, ["x", "residual", "finite"])


target_integrand_tree = canonicalize(parse_prefix_string(OBS_TARGET_INTEGRAND))
gold_antiderivative_tree = canonicalize(parse_prefix_string(OBS_GOLD_ANTIDERIVATIVE))
mutation_demo = mutate_once(
    gold_antiderivative_tree,
    sigma_small=OBS_SIGMA_SMALL,
    rng=random.Random(OBS_MUTATION_SEED),
)
if mutation_demo is None:
    raise RuntimeError("Expected a deterministic forward mutation for the observation demo.")

clean_observation = build_observation(
    target_integrand_tree,
    gold_antiderivative_tree,
    residual_mode="both",
)
mutated_observation = build_observation(
    target_integrand_tree,
    mutation_demo.mutated_expr,
    residual_mode="both",
)

print(f"Target integrand f:      {serialize_prefix_string(target_integrand_tree)}")
print(f"Gold antiderivative I*:  {serialize_prefix_string(gold_antiderivative_tree)}")
print(
    "One current candidate I_t from mutate_once(...): "
    f"{serialize_prefix_string(mutation_demo.mutated_expr)}"
)
print(
    "The observation stores f and the current candidate I_t, "
    "but not the reverse edit label back toward I*."
)

show_observation("Clean observation: I_t already differentiates back to f", clean_observation)
show_observation("Corrupted observation: I_t came from one forward mutation", mutated_observation)


Target integrand f:      add cos x pow x INT+ 2
Gold antiderivative I*:  add sin x div pow x INT+ 3 INT+ 3
One current candidate I_t from mutate_once(...): add sec x div pow x INT+ 3 INT+ 3
The observation stores f and the current candidate I_t, but not the reverse edit label back toward I*.
--------------------------------------------------------------------------------
Clean observation: I_t already differentiates back to f
target integrand:       add cos x pow x INT+ 2
current antiderivative: add sin x div pow x INT+ 3 INT+ 3
current derivative:     add cos x pow x INT+ 2
symbolic residual:      INT+ 0
residual_mode:          both
status:                 ok
warnings:               []
probe points:           (0.25, 0.5, 1.0, 2.0, 3.0, 5.0)
mean_abs_residual:      0.0
mean_squared_residual:  0.0
max_abs_residual:       0.0
fraction_finite:        1.0
x    | residual | finite
-----+----------+-------
0.25 | 0.0      | True  
0.5  | 0.0      | True  
1.0  | 0.0      | True  
2.0  | 0.0 

In [32]:
mode_rows = []
for residual_mode in ("none", "symbolic", "numeric", "both"):
    observation = build_observation(
        target_integrand_tree,
        mutation_demo.mutated_expr,
        residual_mode=residual_mode,
    )
    mode_rows.append(
        {
            "residual_mode": residual_mode,
            "has_derivative": observation.current_derivative is not None,
            "has_symbolic_residual": observation.symbolic_residual is not None,
            "has_numeric_probes": observation.numeric_probes is not None,
            "status": observation.status,
            "warnings": ", ".join(observation.warnings) or "-",
        }
    )

print(f"Default numeric probe points: {DEFAULT_PROBE_POINTS}")
print("Residual modes on the same current candidate:")
print_table(
    mode_rows,
    [
        "residual_mode",
        "has_derivative",
        "has_symbolic_residual",
        "has_numeric_probes",
        "status",
        "warnings",
    ],
)


Default numeric probe points: (0.25, 0.5, 1.0, 2.0, 3.0, 5.0)
Residual modes on the same current candidate:
residual_mode | has_derivative | has_symbolic_residual | has_numeric_probes | status | warnings
--------------+----------------+-----------------------+--------------------+--------+---------
none          | True           | False                 | False              | ok     | -       
symbolic      | True           | True                  | False              | ok     | -       
numeric       | True           | False                 | True               | ok     | -       
both          | True           | True                  | True               | ok     | -       


## Reverse Edit Path

Phase 2 adds the supervised reverse-edit target. This is different from simply undoing the last sampled mutation: given a current corrupted tree and the canonical target tree, `first_edit_toward_target(...)` finds the first useful legal repair step on a clean path back to the target.

That distinction matters for training. Forward mutation is just a way to create a challenging `I_t`; the model should learn a good repair policy from the visible state `(f, I_t)`, not memorize how to reverse a hidden random process it will never see at inference time.

The broad `Expr` subtree rule matters here too. Reverse repair is now free to use direct small cross-shape subtree jumps when they reduce distance to the target, so some paths that previously needed an intermediate same-family repair can now jump straight to the useful target subtree.

Each reverse edit reports the selected preorder node id in the current canonical tree, the canonical token span for that node, the mutation kind, the original subtree, the replacement subtree, and the resulting tree after replacement plus canonicalization.

The first code cell below shows representative reverse edits: exact local repair, direct leaf-to-subtree repair, direct cross-shape subtree repair, and a small target-subtree repair inside a larger tree. The second code cell keeps an end-to-end corruption-and-repair demo using `compute_edit_path(...)`.


In [33]:
from src.tree_diffusion.edit_path import compute_edit_path, first_edit_toward_target, structural_distance


REVERSE_EDIT_DEMOS = [
    {
        "label": "Local constant repair",
        "current": "pow x INT+ 5",
        "target": "pow x INT+ 3",
        "sigma_small": 0,
        "seed": 0,
    },
    {
        "label": "Leaf to non-leaf subtree repair",
        "current": "add x x",
        "target": "add x pow x INT+ 2",
        "sigma_small": 2,
        "seed": 0,
    },
    {
        "label": "Cross-shape subtree repair",
        "current": "sin x",
        "target": "div x INT+ 2",
        "sigma_small": 1,
        "seed": 0,
    },
    {
        "label": "Direct small child-subtree repair",
        "current": "pow x INT+ 5",
        "target": "pow x add x pow x INT+ 2",
        "sigma_small": 2,
        "seed": 0,
    },
]

reverse_rows = []
for demo in REVERSE_EDIT_DEMOS:
    current_tree = canonicalize(parse_prefix_string(demo["current"]))
    target_tree = canonicalize(parse_prefix_string(demo["target"]))
    edit = first_edit_toward_target(
        current_tree,
        target_tree,
        sigma_small=demo["sigma_small"],
        rng=random.Random(demo["seed"]),
    )
    if edit is None:
        raise RuntimeError(f"first_edit_toward_target returned None for {demo}")

    before = structural_distance(current_tree, target_tree)
    after = structural_distance(edit.resulting_tree, target_tree)
    assert after < before, (demo, before, after)

    reverse_rows.append(
        {
            "label": demo["label"],
            "reason": edit.reason,
            "mutation_kind": edit.mutation_kind,
            "sigma_small": demo["sigma_small"],
            "selected_node_id": edit.selected_node_id,
            "selected_node_span": edit.selected_node_span,
            "current": serialize_prefix_string(current_tree),
            "target": serialize_prefix_string(target_tree),
            "original_subtree": serialize_prefix_string(edit.original_subtree),
            "replacement_subtree": serialize_prefix_string(edit.replacement_subtree),
            "resulting_tree": serialize_prefix_string(edit.resulting_tree),
            "distance": f"{before} -> {after}",
        }
    )

print("Representative first_edit_toward_target(...) examples under the broad Expr subtree rule:")
print_table(
    reverse_rows,
    [
        "label",
        "reason",
        "mutation_kind",
        "sigma_small",
        "selected_node_id",
        "selected_node_span",
        "current",
        "target",
        "original_subtree",
        "replacement_subtree",
        "resulting_tree",
        "distance",
    ],
)


Representative first_edit_toward_target(...) examples under the broad Expr subtree rule:
label                             | reason                 | mutation_kind                     | sigma_small | selected_node_id | selected_node_span | current      | target                   | original_subtree | replacement_subtree | resulting_tree           | distance
----------------------------------+------------------------+-----------------------------------+-------------+------------------+--------------------+--------------+--------------------------+------------------+---------------------+--------------------------+---------
Local constant repair             | direct_mismatch_target | local_const_edit                  | 0           | 2                | (2, 4)             | pow x INT+ 5 | pow x INT+ 3             | INT+ 5           | INT+ 3              | pow x INT+ 3             | 1 -> 0  
Leaf to non-leaf subtree repair   | direct_mismatch_target | sampled_small_subtree_replacement | 2   

## Full Reverse Path Demo

The final reverse-edit demo uses a more structured target expression with nested `mul`, `add`, `sin`, `pow`, `exp`, and `ln` nodes. It applies four valid forward mutations with `mutate_once(...)`, then runs `compute_edit_path(...)` to show the sequence of corrective `EditTarget` objects that returns the corrupted tree to the original target.

This section is the first place the full training story becomes visible:

1. Start at a known `I*`.
2. Corrupt it with forward mutations to create `I_t`.
3. Ignore the individual mutation history when creating the label.
4. Ask `compute_edit_path(...)` for useful reverse repairs from `I_t` back to `I*`.

Because sampled subtree replacement is now broad over `Expr`, some reverse steps can repair a whole small subtree in one move instead of walking through a same-family intermediate.


In [34]:
from src.tree_diffusion.edit_path import compute_edit_path, structural_distance


REVERSE_DEMO_EXPRESSION = "mul add x pow x INT+ 2 add sin x add exp x ln x"
REVERSE_DEMO_SIGMA_SMALL = 2
REVERSE_DEMO_MUTATION_STEPS = 4
REVERSE_DEMO_SEED = 0

target_tree = canonicalize(parse_prefix_string(REVERSE_DEMO_EXPRESSION))
current_tree = target_tree
forward_rng = random.Random(REVERSE_DEMO_SEED)
forward_mutations = []

for step in range(REVERSE_DEMO_MUTATION_STEPS):
    mutation = mutate_once(current_tree, sigma_small=REVERSE_DEMO_SIGMA_SMALL, rng=forward_rng)
    if mutation is None:
        raise RuntimeError(f"mutate_once returned None at forward step {step + 1}")
    forward_mutations.append(mutation)
    current_tree = mutation.mutated_expr

reverse_path = compute_edit_path(
    current_tree,
    target_tree,
    REVERSE_DEMO_SIGMA_SMALL,
    rng=random.Random(REVERSE_DEMO_SEED + 100),
    max_steps=32,
)

print("Target tree:    ", serialize_prefix_string(target_tree))
print("Corrupted tree: ", serialize_prefix_string(current_tree))
print("Initial reverse distance:", structural_distance(current_tree, target_tree))
print()
print("Forward mutations:")
for step, mutation in enumerate(forward_mutations, start=1):
    print(
        f"{step}. kind={mutation.mutation_kind} node={mutation.selected_node_id} "
        f"{serialize_prefix_string(mutation.original_subtree)} -> "
        f"{serialize_prefix_string(mutation.replacement_subtree)} "
        f"=> {serialize_prefix_string(mutation.mutated_expr)}"
    )

print()
print("Reverse edit path:")
if not reverse_path:
    print("No reverse edit was needed; canonical trees were already equal.")

for step, edit in enumerate(reverse_path, start=1):
    previous_tree = current_tree if step == 1 else reverse_path[step - 2].resulting_tree
    before = structural_distance(previous_tree, target_tree)
    after = structural_distance(edit.resulting_tree, target_tree)
    print(
        f"{step}. reason={edit.reason} kind={edit.mutation_kind} "
        f"node={edit.selected_node_id} span={edit.selected_node_span} "
        f"{serialize_prefix_string(edit.original_subtree)} -> "
        f"{serialize_prefix_string(edit.replacement_subtree)} "
        f"=> {serialize_prefix_string(edit.resulting_tree)} "
        f"(distance {before} -> {after})"
    )

assert reverse_path, "Expected at least one reverse edit in this demo."
assert reverse_path[-1].resulting_tree == target_tree
print()
print("Recovered target:", reverse_path[-1].resulting_tree == target_tree)


Target tree:     mul add x pow x INT+ 2 add exp x add ln x sin x
Corrupted tree:  mul add x pow x INT+ 2 add csc div x INT- 1 add exp x sin x
Initial reverse distance: 7

Forward mutations:
1. kind=local_same_arity_replacement node=10 ln x -> sec x => mul add x pow x INT+ 2 add exp x add sec x sin x
2. kind=sampled_small_subtree_replacement node=11 x -> div x INT+ 3 => mul add x pow x INT+ 2 add exp x add sec div x INT+ 3 sin x
3. kind=local_const_edit node=13 INT+ 3 -> INT- 1 => mul add x pow x INT+ 2 add exp x add sec div x INT- 1 sin x
4. kind=local_same_arity_replacement node=10 sec div x INT- 1 -> csc div x INT- 1 => mul add x pow x INT+ 2 add csc div x INT- 1 add exp x sin x

Reverse edit path:
1. reason=direct_mismatch_target kind=sampled_small_subtree_replacement node=7 span=(8, 13) csc div x INT- 1 -> exp x => mul add x pow x INT+ 2 add exp x add exp x sin x (distance 7 -> 1)
2. reason=direct_mismatch_target kind=local_same_arity_replacement node=10 span=(11, 13) exp x -> ln x

## Model-Ready Tokenization

At this point we have the two ingredients the model-facing layer needs: an inference-time observation built from `(f, I_t)` and a supervised reverse edit target that points back toward the gold antiderivative `I*`. `TreeDiffusionTokenizer(...)` keeps those separate.

The model input is the serialized observation fields followed by `<EDIT>`. The training label is the next reverse edit only: a position token like `<POS_7>`, the replacement subtree tokens, and `<eos>`. The code below shows both a plain expression tokenization example and the full observation-plus-edit pair that would be ready to batch for a model.

The extra angle-bracket tokens are control tokens. They are not math operators; they mark fields, missing values, sequence boundaries, numeric feature buckets, and edit locations.

| Token or pattern | Meaning |
|---|---|
| `<bos>` | Optional beginning-of-sequence marker added when encoding inputs for sequence models. |
| `<eos>` | End-of-target marker appended to edit labels. |
| `<pad>` | Padding token used when fixed-length batches are requested. |
| `<unk>` | Unknown-token id, only used if encoding explicitly allows unknowns. |
| `<F> ... </F>` | Target integrand `f`. |
| `<CUR> ... </CUR>` | Current antiderivative candidate `I_t`. |
| `<DER> ... </DER>` | Current derivative `g_t = d/dx I_t`, or `<NO_DER>` if unavailable. |
| `<RES> ... </RES>` | Symbolic residual `g_t - f`, or `<NO_RES>` if unavailable or disabled. |
| `<NUM> ... </NUM>` | Numeric probe residual features, or `<NO_NUM>` if disabled or unavailable. |
| `<NUM_VALUE_i>` | Marks the bucketed residual value at probe index `i`. |
| `<NUM_MEAN_ABS>`, `<NUM_MSE>`, `<NUM_MAX_ABS>` | Mark aggregate numeric residual features. |
| `<NUM_ZERO>`, `<NUM_NAN>`, `<NUM_POS_LOG_k>`, `<NUM_NEG_LOG_k>` | Coarse buckets for numeric values by sign and base-10 magnitude. |
| `<EDIT>` | Separates the serialized observation from the edit the model should now predict. |
| `<POS_i>` | Target-side pointer to preorder `node_id == i` in the current canonical tree. |

So a training row should be read as: "given fields `<F>`, `<CUR>`, `<DER>`, `<RES>`, and `<NUM>`, predict an edit after `<EDIT>`; the answer starts by pointing at one current-tree node with `<POS_i>`."


In [35]:
reverse_demo_integrand = compute_current_derivative(target_tree)
reverse_demo_observation = build_observation(
    reverse_demo_integrand,
    current_tree,
    residual_mode="both",
)
next_edit = reverse_path[0]
tokenizer = TreeDiffusionTokenizer(max_positions=512)

CONTROL_TOKEN_DEMOS = [
    {"token": "<F>...</F>", "section": "target integrand f"},
    {"token": "<CUR>...</CUR>", "section": "current antiderivative I_t"},
    {"token": "<DER>...</DER>", "section": "current derivative g_t"},
    {"token": "<RES>...</RES>", "section": "symbolic residual g_t - f"},
    {"token": "<NUM>...</NUM>", "section": "numeric residual probes"},
    {"token": "<EDIT>", "section": "input/target boundary"},
    {"token": tokenizer.position_token(next_edit.selected_node_id), "section": "selected node pointer"},
]
print("Tokenizer vocabulary size:", tokenizer.vocabulary_size)
print("Control tokens used by this example:")
print_table(CONTROL_TOKEN_DEMOS, ["token", "section"])
print()

expression_tokens = tokenizer.serialize_expr(current_tree)
expression_ids = tokenizer.encode_tokens(expression_tokens)
input_tokens, target_tokens = tokenizer.serialize_training_pair(reverse_demo_observation, next_edit)
input_ids = tokenizer.encode_tokens(input_tokens, add_bos=True)
target_ids = tokenizer.encode_tokens(target_tokens)

assert tokenizer.decode_ids(expression_ids) == expression_tokens
assert tokenizer.decode_ids(input_ids)[0] == tokenizer.bos_token
assert tokenizer.decode_ids(input_ids)[1:] == input_tokens
assert target_tokens[0] == tokenizer.position_token(next_edit.selected_node_id)
assert target_tokens[-1] == tokenizer.eos_token

print("Integrand f:                 ", serialize_prefix_string(reverse_demo_integrand))
print("Current antiderivative I_t:  ", serialize_prefix_string(current_tree))
print("Gold antiderivative I*:      ", serialize_prefix_string(target_tree))
print()
print("Plain expression tokenization for I_t:")
print(expression_tokens)
print(expression_ids)
print()
print("Model input tokens (with <bos>):")
print(tokenizer.decode_ids(input_ids))
print(input_ids)
print()
print("Model target tokens:")
print(target_tokens)
print(target_ids)


Tokenizer vocabulary size: 661
Control tokens used by this example:
token          | section                   
---------------+---------------------------
<F>...</F>     | target integrand f        
<CUR>...</CUR> | current antiderivative I_t
<DER>...</DER> | current derivative g_t    
<RES>...</RES> | symbolic residual g_t - f 
<NUM>...</NUM> | numeric residual probes   
<EDIT>         | input/target boundary     
<POS_7>        | selected node pointer     

Integrand f:                  add mul add INT+ 1 mul INT+ 2 x add exp x add ln x sin x mul add x pow x INT+ 2 add cos x add exp x pow x INT- 1
Current antiderivative I_t:   mul add x pow x INT+ 2 add csc div x INT- 1 add exp x sin x
Gold antiderivative I*:       mul add x pow x INT+ 2 add exp x add ln x sin x

Plain expression tokenization for I_t:
['mul', 'add', 'x', 'pow', 'x', 'INT+', '2', 'add', 'csc', 'div', 'x', 'INT-', '1', 'add', 'exp', 'x', 'sin', 'x']
[86, 72, 96, 87, 96, 65, 55, 72, 81, 83, 96, 66, 54, 72, 84, 96, 91, 

## Supervised Training Example Generation

The previous sections built the pieces separately: forward corruption, observations, reverse edit paths, and tokenization. The training-example generator wires those pieces together in the tree-diffusion style:

1. Start from the canonical gold antiderivative `I*`.
2. Produce a current candidate `I_t` either by applying small forward mutations or by random initialization with probability `rho`.
3. Compute the first useful reverse edit from `I_t` toward `I*` with `first_edit_toward_target(...)`.
4. Serialize only `Observation(target_integrand=f, current_antiderivative=I_t) + ["<EDIT>"]` as model input.
5. Serialize the label as `<POS_i> replacement_subtree <eos>`.

The important supervision invariant is that the label is a reverse-path edit toward the target tree, not simply the inverse of the last sampled forward mutation. The gold antiderivative is metadata for debugging, auditing, and computing the label; it is not added as a dedicated input field.

Key generation knobs:

| Parameter | Meaning |
|---|---|
| `sigma_small` | Maximum operator-node size for small mutable/replacement subtrees. |
| `smax` | Maximum number of forward mutation steps used when building a corrupted candidate. |
| `rho` | Probability of ignoring mutation mode and starting from a bounded random expression instead. |
| `max_random_size` | Size budget for random initialization when supplied. |
| `residual_mode` | Which residual fields are included in the observation: `none`, `symbolic`, `numeric`, or `both`. |
| `encode` | Whether to convert serialized tokens to padded token ids immediately. |

The helper assertions below keep checking that no hidden gold-field token enters the input sequence.


In [36]:
from src.tree_diffusion.training_examples import (
    TreeDiffusionTrainingExample,
    generate_current_candidate,
    generate_training_example,
)
from src.tree_diffusion.mutation import sample_random_expr


TRAINING_TOKENIZER = TreeDiffusionTokenizer(max_positions=512)
FORBIDDEN_GOLD_FIELD_TOKENS = {"<TARGET>", "<GOLD>", "<ANTIDERIVATIVE_TARGET>", "<I_STAR>"}


def token_section(tokens: list[str], start_token: str, end_token: str) -> list[str]:
    start = tokens.index(start_token) + 1
    end = tokens.index(end_token, start)
    return tokens[start:end]


def compact(text: str, width: int = 90) -> str:
    return text if len(text) <= width else text[: width - 3] + "..."


def summarize_training_example(
    label: str,
    example: TreeDiffusionTrainingExample,
    *,
    tokenizer: TreeDiffusionTokenizer = TRAINING_TOKENIZER,
    show_tokens: bool = True,
) -> None:
    before = structural_distance(example.current_antiderivative, example.target_antiderivative)
    after = structural_distance(example.edit_target.resulting_tree, example.target_antiderivative)
    position = tokenizer.token_to_position(example.target_tokens[0])
    current_index = index_tree_positions(example.current_antiderivative)
    assert position in current_index.node_id_to_node
    assert not (FORBIDDEN_GOLD_FIELD_TOKENS & set(example.input_tokens))

    print("=" * 100)
    print(label)
    print(f"target integrand f:       {serialize_prefix_string(example.target_integrand)}")
    print(f"gold antiderivative I*:   {serialize_prefix_string(example.target_antiderivative)}")
    print(f"current candidate I_t:    {serialize_prefix_string(example.current_antiderivative)}")
    print(f"used_random_init:         {example.used_random_init}")
    print(f"num_mutations:            {example.num_mutations}")
    print(f"attempts:                 {example.attempts}")
    print(f"observation status:       {example.observation.status}")
    print(f"observation warnings:     {list(example.warnings)}")
    print(f"edit reason:              {example.edit_target.reason}")
    print(f"edit kind:                {example.edit_target.mutation_kind}")
    print(f"edit position token:      {example.target_tokens[0]} -> node_id={position}")
    print(f"original subtree:         {serialize_prefix_string(example.edit_target.original_subtree)}")
    print(f"replacement subtree:      {serialize_prefix_string(example.edit_target.replacement_subtree)}")
    print(f"resulting tree:           {serialize_prefix_string(example.edit_target.resulting_tree)}")
    print(f"structural distance:      {before} -> {after}")
    print(f"input token length:       {len(example.input_tokens)}")
    print(f"target token length:      {len(example.target_tokens)}")
    print(f"input ids present:        {example.input_ids is not None}")
    print(f"target ids present:       {example.target_ids is not None}")
    if show_tokens:
        print("input tokens:")
        print(example.input_tokens)
        print("target tokens:")
        print(example.target_tokens)


### Candidate Generation: Mutation Mode, Random Initialization, And Direct Random Trees

`generate_current_candidate(...)` is the forward/noising side of the pipeline. With `rho=0`, every candidate comes from one or more `mutate_once(...)` steps applied to the gold antiderivative. With `rho=1`, every candidate is sampled directly from the bounded random expression sampler. Intermediate `rho` mixes the two modes.

`num_mutations` records how many successful forward mutations were applied. Random initialization uses `num_mutations=0` because it did not walk away from `I*`; it jumped directly to an independently sampled expression.


In [37]:
TRAINING_TARGET_ANTIDERIVATIVE = canonicalize(parse_prefix_string("add div pow x INT+ 3 INT+ 3 sin x"))

candidate_rows = []
for seed, rho, smax, max_random_size in [
    (0, 0.0, 1, None),
    (1, 0.0, 3, None),
    (2, 0.0, 5, None),
    (3, 0.5, 4, 3),
    (4, 0.5, 4, 3),
    (5, 1.0, 4, 2),
    (6, 1.0, 4, 3),
    (7, 1.0, 4, 4),
]:
    current, num_mutations, used_random_init = generate_current_candidate(
        TRAINING_TARGET_ANTIDERIVATIVE,
        rng=random.Random(seed),
        sigma_small=2,
        smax=smax,
        rho=rho,
        max_random_size=max_random_size,
    )
    candidate_rows.append(
        {
            "seed": seed,
            "rho": rho,
            "mode": "random-init" if used_random_init else "mutations",
            "num_mutations": num_mutations,
            "distance_to_I*": structural_distance(current, TRAINING_TARGET_ANTIDERIVATIVE),
            "candidate": compact(serialize_prefix_string(current), 110),
        }
    )

print("Candidates generated from one gold antiderivative:")
print_table(candidate_rows, ["seed", "rho", "mode", "num_mutations", "distance_to_I*", "candidate"])

random_tree_rows = []
for max_size in range(5):
    for seed in range(2):
        sampled = sample_random_expr(rng=random.Random(1000 + 10 * max_size + seed), max_size=max_size)
        random_tree_rows.append(
            {
                "max_size": max_size,
                "seed": seed,
                "sampled_expr": serialize_prefix_string(sampled),
            }
        )

print("\nDirect bounded random expression samples used by random initialization:")
print_table(random_tree_rows, ["max_size", "seed", "sampled_expr"])


Candidates generated from one gold antiderivative:
seed | rho | mode        | num_mutations | distance_to_I* | candidate                                                     
-----+-----+-------------+---------------+----------------+---------------------------------------------------------------
0    | 0.0 | mutations   | 1             | 8              | add div div INT+ 1 INT+ 3 div INT+ 1 x div pow x INT+ 3 INT+ 3
1    | 0.0 | mutations   | 1             | 14             | div pow x INT+ 3 INT+ 3                                       
2    | 0.0 | mutations   | 1             | 1              | add sin x div pow x div INT+ 1 INT+ 2 INT+ 3                  
3    | 0.5 | random-init | 0             | 15             | sinh div INT- 1 pow I INT+ 1                                  
4    | 0.5 | random-init | 0             | 10             | INT+ 0                                                        
5    | 1.0 | random-init | 0             | 11             | atanh INT+ 3                

### Full Training Examples From Hand-Picked Integration Pairs

Each row below is one complete supervised example. Notice that the model input is based on the target integrand `f` and the current candidate `I_t`; `I*` is shown here only because this is a debugging walkthrough.

For each example, the target token sequence should be read as "edit this current tree at `<POS_i>` by replacing that selected subtree with these prefix tokens, then stop at `<eos>`." The replacement is chosen because it moves the current candidate toward `I*` under the reverse edit policy.


In [38]:
TRAINING_PAIR_DEMOS = [
    {
        "label": "polynomial power rule",
        "f": "pow x INT+ 2",
        "I_star": "div pow x INT+ 3 INT+ 3",
        "seed": 10,
        "rho": 0.0,
    },
    {
        "label": "trig cosine",
        "f": "cos x",
        "I_star": "sin x",
        "seed": 11,
        "rho": 0.0,
    },
    {
        "label": "exponential fixed point",
        "f": "exp x",
        "I_star": "exp x",
        "seed": 12,
        "rho": 0.0,
    },
    {
        "label": "sum: polynomial plus sine",
        "f": "add pow x INT+ 2 cos x",
        "I_star": "add div pow x INT+ 3 INT+ 3 sin x",
        "seed": 13,
        "rho": 0.0,
    },
    {
        "label": "log antiderivative",
        "f": "div INT+ 1 x",
        "I_star": "ln x",
        "seed": 14,
        "rho": 0.0,
    },
    {
        "label": "chain rule sine x^2",
        "f": "mul INT+ 2 mul x cos pow x INT+ 2",
        "I_star": "sin pow x INT+ 2",
        "seed": 15,
        "rho": 0.0,
    },
    {
        "label": "mixed target with random init",
        "f": "add sin x pow x INT+ 2",
        "I_star": "add mul INT- 1 cos x div pow x INT+ 3 INT+ 3",
        "seed": 16,
        "rho": 1.0,
    },
    {
        "label": "random init for polynomial",
        "f": "mul INT+ 3 pow x INT+ 2",
        "I_star": "pow x INT+ 3",
        "seed": 17,
        "rho": 1.0,
    },
]

training_examples = []
summary_rows = []
for demo in TRAINING_PAIR_DEMOS:
    example = generate_training_example(
        parse_prefix_string(demo["f"]),
        parse_prefix_string(demo["I_star"]),
        tokenizer=TRAINING_TOKENIZER,
        rng=random.Random(demo["seed"]),
        sigma_small=2,
        smax=3,
        rho=demo["rho"],
        residual_mode="both",
    )
    training_examples.append((demo, example))
    before = structural_distance(example.current_antiderivative, example.target_antiderivative)
    after = structural_distance(example.edit_target.resulting_tree, example.target_antiderivative)
    summary_rows.append(
        {
            "label": demo["label"],
            "rho": demo["rho"],
            "mode": "random" if example.used_random_init else "mutated",
            "mutations": example.num_mutations,
            "distance": f"{before}->{after}",
            "position": example.target_tokens[0],
            "replacement": serialize_prefix_string(example.edit_target.replacement_subtree),
            "input_len": len(example.input_tokens),
            "target_len": len(example.target_tokens),
        }
    )

print_table(
    summary_rows,
    ["label", "rho", "mode", "mutations", "distance", "position", "replacement", "input_len", "target_len"],
)

for demo, example in training_examples[:4]:
    summarize_training_example(demo["label"], example, show_tokens=False)


label                         | rho | mode    | mutations | distance | position | replacement             | input_len | target_len
------------------------------+-----+---------+-----------+----------+----------+-------------------------+-----------+-----------
polynomial power rule         | 0.0 | mutated | 3         | 7->0     | <POS_0>  | div pow x INT+ 3 INT+ 3 | 44        | 9         
trig cosine                   | 0.0 | mutated | 2         | 4->0     | <POS_0>  | sin x                   | 41        | 4         
exponential fixed point       | 0.0 | mutated | 2         | 6->0     | <POS_0>  | exp x                   | 46        | 4         
sum: polynomial plus sine     | 0.0 | mutated | 2         | 5->0     | <POS_4>  | pow x INT+ 3            | 64        | 6         
log antiderivative            | 0.0 | mutated | 1         | 4->0     | <POS_0>  | ln x                    | 47        | 4         
chain rule sine x^2           | 0.0 | mutated | 1         | 1->0     | <POS_3>  | I

### Model-Ready Encoded Example

When `encode=True`, the same serialized pair is converted to token ids. Padding is handled by the tokenizer, and decoding with `strip_pad=True` recovers the exact serialized tokens.

Input ids and target ids are separate sequences. Padding tokens keep every row in a batch the same length; attention masks later tell the model which positions are real tokens and which positions are padding.


In [39]:
encoded_example = generate_training_example(
    parse_prefix_string("pow x INT+ 2"),
    parse_prefix_string("div pow x INT+ 3 INT+ 3"),
    tokenizer=TRAINING_TOKENIZER,
    rng=random.Random(2024),
    sigma_small=2,
    smax=3,
    rho=0.0,
    residual_mode="both",
    encode=True,
    max_input_length=256,
    max_target_length=64,
)

assert encoded_example.input_ids is not None
assert encoded_example.target_ids is not None
assert len(encoded_example.input_ids) == 256
assert len(encoded_example.target_ids) == 64
assert TRAINING_TOKENIZER.decode_ids(encoded_example.input_ids, strip_pad=True) == encoded_example.input_tokens
assert TRAINING_TOKENIZER.decode_ids(encoded_example.target_ids, strip_pad=True) == encoded_example.target_tokens
assert encoded_example.input_tokens[-1] == "<EDIT>"
assert encoded_example.target_tokens[-1] == TRAINING_TOKENIZER.eos_token

summarize_training_example("encoded polynomial example", encoded_example, show_tokens=True)
print("encoded input ids:")
print(encoded_example.input_ids)
print("encoded target ids:")
print(encoded_example.target_ids)


encoded polynomial example
target integrand f:       pow x INT+ 2
gold antiderivative I*:   div pow x INT+ 3 INT+ 3
current candidate I_t:    pow div x INT+ 3 INT+ 3
used_random_init:         False
num_mutations:            2
attempts:                 1
observation status:       ok
observation warnings:     []
edit reason:              direct_mismatch_target
edit kind:                sampled_small_subtree_replacement
edit position token:      <POS_0> -> node_id=0
original subtree:         pow div x INT+ 3 INT+ 3
replacement subtree:      div pow x INT+ 3 INT+ 3
resulting tree:           div pow x INT+ 3 INT+ 3
structural distance:      2 -> 0
input token length:       60
target token length:      9
input ids present:        True
target ids present:       True
input tokens:
['<F>', 'pow', 'x', 'INT+', '2', '</F>', '<CUR>', 'pow', 'div', 'x', 'INT+', '3', 'INT+', '3', '</CUR>', '<DER>', 'mul', 'div', 'INT+', '1', 'INT+', '9', 'pow', 'x', 'INT+', '2', '</DER>', '<RES>', 'mul', 'div', 'INT

### Residual Modes In Generated Examples

The generator forwards `residual_mode` into `build_observation(...)`, so the same training-pair format can be used with no residuals, symbolic residuals, numeric probes, or both.

The field wrappers remain stable even when a component is disabled. For example, disabled symbolic residuals serialize as `<RES> <NO_RES> </RES>`, and disabled numeric probes serialize as `<NUM> <NO_NUM> </NUM>`. Keeping the wrappers stable makes the input layout easier for downstream code to parse and batch.


In [40]:
residual_mode_rows = []
for residual_mode in ("none", "symbolic", "numeric", "both"):
    example = generate_training_example(
        parse_prefix_string("pow x INT+ 2"),
        parse_prefix_string("div pow x INT+ 3 INT+ 3"),
        tokenizer=TRAINING_TOKENIZER,
        rng=random.Random(300),
        sigma_small=2,
        smax=2,
        rho=0.0,
        residual_mode=residual_mode,
    )
    residual_tokens = token_section(example.input_tokens, "<RES>", "</RES>")
    numeric_tokens = token_section(example.input_tokens, "<NUM>", "</NUM>")
    residual_mode_rows.append(
        {
            "mode": residual_mode,
            "status": example.observation.status,
            "residual_tokens": compact(" ".join(residual_tokens), 80),
            "numeric_tokens": compact(" ".join(numeric_tokens), 80),
            "input_len": len(example.input_tokens),
            "warnings": list(example.warnings),
        }
    )

print_table(
    residual_mode_rows,
    ["mode", "status", "residual_tokens", "numeric_tokens", "input_len", "warnings"],
)


mode     | status | residual_tokens         | numeric_tokens                                                                   | input_len | warnings
---------+--------+-------------------------+----------------------------------------------------------------------------------+-----------+---------
none     | ok     | <NO_RES>                | <NO_NUM>                                                                         | 24        | []      
symbolic | ok     | mul INT- 1 pow x INT+ 2 | <NO_NUM>                                                                         | 30        | []      
numeric  | ok     | <NO_RES>                | <NUM_VALUE_0> <NUM_NEG_LOG_-2> <NUM_VALUE_1> <NUM_NEG_LOG_-1> <NUM_VALUE_2> <... | 41        | []      
both     | ok     | mul INT- 1 pow x INT+ 2 | <NUM_VALUE_0> <NUM_NEG_LOG_-2> <NUM_VALUE_1> <NUM_NEG_LOG_-1> <NUM_VALUE_2> <... | 47        | []      


### Reverse Path Audit For A Generated Example

The generated label should be a useful edit on the path from the current candidate toward `I*`. The cell below recomputes a short reverse path from the generated current tree and compares the training target to the path's first move.

This is a sanity check on the key training invariant: the target tokens are a next repair edit from the visible current tree, not a hidden record of how that tree was corrupted.


In [41]:
path_audit_example = generate_training_example(
    parse_prefix_string("add pow x INT+ 2 cos x"),
    parse_prefix_string("add div pow x INT+ 3 INT+ 3 sin x"),
    tokenizer=TRAINING_TOKENIZER,
    rng=random.Random(444),
    sigma_small=2,
    smax=5,
    rho=0.0,
    residual_mode="both",
)
path = compute_edit_path(
    path_audit_example.current_antiderivative,
    path_audit_example.target_antiderivative,
    sigma_small=2,
    rng=random.Random(444),
    max_steps=16,
)

summarize_training_example("reverse path audit example", path_audit_example, show_tokens=False)
print(f"computed reverse path length: {len(path)}")
for step_index, edit in enumerate(path[:8]):
    before = structural_distance(
        path_audit_example.current_antiderivative if step_index == 0 else path[step_index - 1].resulting_tree,
        path_audit_example.target_antiderivative,
    )
    after = structural_distance(edit.resulting_tree, path_audit_example.target_antiderivative)
    print(
        f"step {step_index}: node={edit.selected_node_id}, kind={edit.mutation_kind}, "
        f"reason={edit.reason}, replacement={serialize_prefix_string(edit.replacement_subtree)}, "
        f"distance={before}->{after}"
    )

print("\ntraining target tokens:")
print(path_audit_example.target_tokens)


reverse path audit example
target integrand f:       add cos x pow x INT+ 2
gold antiderivative I*:   add sin x div pow x INT+ 3 INT+ 3
current candidate I_t:    add sin x div asinh add INT+ 1 x x
used_random_init:         False
num_mutations:            2
attempts:                 2
observation status:       ok
observation warnings:     []
edit reason:              direct_mismatch_target
edit kind:                sampled_small_subtree_replacement
edit position token:      <POS_4> -> node_id=4
original subtree:         asinh add INT+ 1 x
replacement subtree:      pow x INT+ 3
resulting tree:           add sin x div pow x INT+ 3 x
structural distance:      11 -> 3
input token length:       127
target token length:      6
input ids present:        False
target ids present:       False
computed reverse path length: 2
step 0: node=4, kind=sampled_small_subtree_replacement, reason=direct_mismatch_target, replacement=pow x INT+ 3, distance=11->3
step 1: node=7, kind=local_same_arity_replacem

### Small Dataset Batch Smoke/Audit

This last single-example audit samples a lightweight batch of real processed dataset pairs, generates encoded training examples, and prints aggregate lengths and edit-distance movement. It intentionally filters out very long prefix rows so the notebook stays responsive with `max_input_length=512`.

This section is optional in local environments: if the processed parquet artifact is missing, the cell prints that fact and continues. When the artifact exists, the rows exercise the same generation path used above on less hand-picked expressions.


In [42]:
DATASET_PATH = REPO_ROOT / "data" / "processed" / "train_prefix_filtered.parquet"
DATASET_BATCH_EXAMPLES = 16
DATASET_READ_ROWS = 500
DATASET_MAX_PREFIX_TOKENS = 40

if not DATASET_PATH.exists():
    print(f"Dataset artifact not found: {DATASET_PATH}")
else:
    import pandas as pd

    raw_dataset_rows = pd.read_parquet(
        DATASET_PATH,
        columns=["integrand_prefix", "integral_prefix"],
    ).head(DATASET_READ_ROWS)
    lightweight_rows = [
        row
        for row in raw_dataset_rows.itertuples(index=False)
        if len(str(row.integrand_prefix).split()) <= DATASET_MAX_PREFIX_TOKENS
        and len(str(row.integral_prefix).split()) <= DATASET_MAX_PREFIX_TOKENS
    ][:DATASET_BATCH_EXAMPLES]

    dataset_examples = []
    dataset_rows = []
    for row_index, row in enumerate(lightweight_rows):
        example = generate_training_example(
            parse_prefix_string(str(row.integrand_prefix)),
            parse_prefix_string(str(row.integral_prefix)),
            tokenizer=TRAINING_TOKENIZER,
            rng=random.Random(50_000 + row_index),
            sigma_small=2,
            smax=3,
            rho=0.2,
            residual_mode="both",
            encode=True,
            max_input_length=512,
            max_target_length=128,
        )
        before = structural_distance(example.current_antiderivative, example.target_antiderivative)
        after = structural_distance(example.edit_target.resulting_tree, example.target_antiderivative)
        dataset_examples.append(example)
        dataset_rows.append(
            {
                "row": row_index,
                "mode": "random" if example.used_random_init else "mutated",
                "mutations": example.num_mutations,
                "distance": f"{before}->{after}",
                "input_len": len(example.input_tokens),
                "target_len": len(example.target_tokens),
                "position": example.target_tokens[0],
                "replacement": compact(serialize_prefix_string(example.edit_target.replacement_subtree), 60),
            }
        )

    print(f"Generated {len(dataset_examples)} encoded examples from lightweight dataset rows.")
    if dataset_examples:
        input_lengths = [len(example.input_tokens) for example in dataset_examples]
        target_lengths = [len(example.target_tokens) for example in dataset_examples]
        random_fraction = sum(example.used_random_init for example in dataset_examples) / len(dataset_examples)
        print(f"average input token length:  {sum(input_lengths) / len(input_lengths):.1f}")
        print(f"average target token length: {sum(target_lengths) / len(target_lengths):.1f}")
        print(f"max input token length:      {max(input_lengths)}")
        print(f"max target token length:     {max(target_lengths)}")
        print(f"fraction random init:        {random_fraction:.2f}")
        print_table(
            dataset_rows,
            ["row", "mode", "mutations", "distance", "input_len", "target_len", "position", "replacement"],
        )

        print("\nFirst few dataset examples, compact view:")
        for row, example in zip(dataset_rows[:4], dataset_examples[:4]):
            print("-" * 100)
            print(f"row {row['row']}: f={compact(serialize_prefix_string(example.target_integrand), 100)}")
            print(f"row {row['row']}: I*={compact(serialize_prefix_string(example.target_antiderivative), 100)}")
            print(f"row {row['row']}: I_t={compact(serialize_prefix_string(example.current_antiderivative), 100)}")
            print(f"row {row['row']}: label={example.target_tokens}")


Generated 16 encoded examples from lightweight dataset rows.
average input token length:  153.0
average target token length: 4.8
max input token length:      415
max target token length:     7
fraction random init:        0.12
row | mode    | mutations | distance | input_len | target_len | position | replacement     
----+---------+-----------+----------+-----------+------------+----------+-----------------
0   | mutated | 1         | 3->0     | 82        | 3          | <POS_6>  | x               
1   | mutated | 3         | 15->8    | 188       | 6          | <POS_4>  | pow x INT+ 2    
2   | mutated | 1         | 6->0     | 97        | 7          | <POS_12> | sin add INT+ 5 x
3   | mutated | 3         | 7->0     | 109       | 3          | <POS_4>  | x               
4   | mutated | 3         | 13->9    | 68        | 6          | <POS_1>  | exp INT+ 1 8    
5   | mutated | 3         | 13->6    | 107       | 6          | <POS_6>  | pow x INT+ 2    
6   | mutated | 2         | 4->0     

### Putting Training Examples Into A Dataloader

The dataset layer wraps the one-example generator into an infinite PyTorch stream. There are two useful access patterns:

1. Iterate the `TreeDiffusionIterableDataset` directly to inspect one already-tokenized, padded item dictionary.
2. Use `make_tree_diffusion_dataloader(...)` to get collated batch tensors for model code.

The dataloader batch contract is the future model-facing surface: `input_ids`, `input_attention_mask`, `target_ids`, `target_attention_mask`, and `labels`, where `labels` is the target sequence with pad positions replaced by `-100`.

Important dataloader details:

| Field or argument | Meaning |
|---|---|
| `input_ids`, `target_ids` | Padded integer token ids. |
| `input_attention_mask`, `target_attention_mask` | `1` for real tokens and `0` for padding. |
| `labels` | Copy of `target_ids` with pad positions replaced by `-100`, the common ignore index for token losses. |
| metadata fields | Human-readable tokens and prefix strings kept as Python lists when `include_metadata=True`. |
| infinite dataset | `TreeDiffusionIterableDataset` keeps yielding generated examples instead of having a fixed epoch length. |
| `base_seed` | Seeds the dataset RNG; worker ids offset that seed for multi-worker loading. |
| `shuffle_pairs` | Chooses random source pairs when true, or cycles through pairs in order when false. |


In [43]:
from src.tree_diffusion.dataset import (
    IntegrationPair,
    TreeDiffusionIterableDataset,
    make_tree_diffusion_dataloader,
    pairs_from_prefix_rows,
)


DATALOADER_PREFIX_ROWS = [
    {
        "index": 0,
        "integrand_prefix": "pow x INT+ 2",
        "integral_prefix": "div pow x INT+ 3 INT+ 3",
    },
    {
        "index": 1,
        "integrand_prefix": "cos x",
        "integral_prefix": "sin x",
    },
    {
        "index": 2,
        "integrand_prefix": "exp x",
        "integral_prefix": "exp x",
    },
    {
        "index": 3,
        "integrand_prefix": "add pow x INT+ 2 cos x",
        "integral_prefix": "add div pow x INT+ 3 INT+ 3 sin x",
    },
]

loader_pairs = pairs_from_prefix_rows(
    DATALOADER_PREFIX_ROWS,
    source="notebook-hand-built",
)
print(f"Built {len(loader_pairs)} IntegrationPair objects for the dataloader demo.")
for pair in loader_pairs:
    print(
        f"pair_index={pair.index}: "
        f"f={serialize_prefix_string(pair.target_integrand)} | "
        f"I*={serialize_prefix_string(pair.target_antiderivative)}"
    )

stream_dataset = TreeDiffusionIterableDataset(
    loader_pairs,
    tokenizer=TRAINING_TOKENIZER,
    sigma_small=2,
    smax=2,
    rho=0.0,
    residual_mode="both",
    max_input_length=256,
    max_target_length=64,
    base_seed=700,
    shuffle_pairs=False,
    include_metadata=True,
)

stream_iterator = iter(stream_dataset)
first_item = next(stream_iterator)
second_item = next(stream_iterator)

print("\nOne dataset item is a plain dictionary. Tensor fields are already padded:")
for key in [
    "input_ids",
    "input_attention_mask",
    "target_ids",
    "target_attention_mask",
    "labels",
]:
    value = first_item[key]
    print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype}")

print("\nScalar fields and metadata from the first two streamed items:")
for label, item in [("first", first_item), ("second", second_item)]:
    print(
        f"{label:6s} pair_index={int(item['pair_index'])}, "
        f"used_random_init={bool(item['used_random_init'])}, "
        f"num_mutations={int(item['num_mutations'])}, "
        f"input_length={int(item['input_length'])}, "
        f"target_length={int(item['target_length'])}"
    )
    print(f"  current_prefix: {item['current_prefix']}")
    print(f"  target_tokens:  {item['target_tokens']}")

print("\nDecoded first item input equals metadata tokens:")
print(TRAINING_TOKENIZER.decode_ids(first_item["input_ids"].tolist(), strip_pad=True))
print(first_item["input_tokens"])
assert TRAINING_TOKENIZER.decode_ids(first_item["input_ids"].tolist(), strip_pad=True) == first_item["input_tokens"]
assert TRAINING_TOKENIZER.decode_ids(first_item["target_ids"].tolist(), strip_pad=True) == first_item["target_tokens"]

label_rows = []
for token_id, label_id, mask in zip(
    first_item["target_ids"][:12].tolist(),
    first_item["labels"][:12].tolist(),
    first_item["target_attention_mask"][:12].tolist(),
):
    label_rows.append(
        {
            "target_token": TRAINING_TOKENIZER.decode_ids([token_id])[0],
            "target_id": token_id,
            "label_id": label_id,
            "target_mask": mask,
        }
    )
print("\nFirst target positions: labels match target ids until padding, then become -100.")
print_table(label_rows, ["target_token", "target_id", "label_id", "target_mask"])


Built 4 IntegrationPair objects for the dataloader demo.
pair_index=0: f=pow x INT+ 2 | I*=div pow x INT+ 3 INT+ 3
pair_index=1: f=cos x | I*=sin x
pair_index=2: f=exp x | I*=exp x
pair_index=3: f=add cos x pow x INT+ 2 | I*=add sin x div pow x INT+ 3 INT+ 3

One dataset item is a plain dictionary. Tensor fields are already padded:
input_ids                shape=(256,) dtype=torch.int64
input_attention_mask     shape=(256,) dtype=torch.int64
target_ids               shape=(64,) dtype=torch.int64
target_attention_mask    shape=(64,) dtype=torch.int64
labels                   shape=(64,) dtype=torch.int64

Scalar fields and metadata from the first two streamed items:
first  pair_index=0, used_random_init=False, num_mutations=1, input_length=62, target_length=9
  current_prefix: pow pow x INT+ 3 INT+ 3
  target_tokens:  ['<POS_0>', 'div', 'pow', 'x', 'INT+', '3', 'INT+', '3', '<eos>']
second pair_index=1, used_random_init=False, num_mutations=2, input_length=40, target_length=4
  current_

### Accessing A Collated Batch

`make_tree_diffusion_dataloader(...)` uses the same infinite dataset, but asks PyTorch to collect multiple item dictionaries and pass them through `TreeDiffusionBatchCollator`. Tensor fields become rank-2 tensors with batch dimension first, while metadata stays as Python lists so it remains easy to inspect.

The last table focuses on the padding boundary in one target row. Before the boundary, `label_id` matches `target_id`; after the boundary, `target_id` is `<pad>` and `label_id` is `-100`, so padded positions do not contribute to the target-token loss.


In [44]:
notebook_loader = make_tree_diffusion_dataloader(
    loader_pairs,
    tokenizer=TRAINING_TOKENIZER,
    batch_size=3,
    num_workers=0,
    sigma_small=2,
    smax=2,
    rho=0.2,
    residual_mode="both",
    max_input_length=256,
    max_target_length=64,
    base_seed=900,
    shuffle_pairs=False,
    include_metadata=True,
)

batch = next(iter(notebook_loader))

print("Batch tensor fields:")
for key in [
    "input_ids",
    "input_attention_mask",
    "target_ids",
    "target_attention_mask",
    "labels",
    "num_mutations",
    "used_random_init",
    "pair_index",
    "input_length",
    "target_length",
]:
    value = batch[key]
    print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} values={value.tolist() if value.ndim == 1 else '<matrix>'}")

print("\nMetadata lists have one entry per batch row:")
for key in ["input_tokens", "target_tokens", "current_prefix", "target_integrand_prefix", "target_antiderivative_prefix", "warning_count"]:
    print(f"{key:30s} len={len(batch[key])}")

row_index = 0
row_input_ids = batch["input_ids"][row_index]
row_target_ids = batch["target_ids"][row_index]
row_labels = batch["labels"][row_index]
row_input_tokens = batch["input_tokens"][row_index]
row_target_tokens = batch["target_tokens"][row_index]

print("\nAccessing batch row 0:")
print(f"pair_index:              {int(batch['pair_index'][row_index])}")
print(f"used_random_init:        {bool(batch['used_random_init'][row_index])}")
print(f"num_mutations:           {int(batch['num_mutations'][row_index])}")
print(f"current_prefix:          {batch['current_prefix'][row_index]}")
print(f"target_integrand_prefix: {batch['target_integrand_prefix'][row_index]}")
print(f"target_antiderivative:   {batch['target_antiderivative_prefix'][row_index]}")
print(f"input tokens end:        {row_input_tokens[-8:]}")
print(f"target tokens:           {row_target_tokens}")
print(f"decoded input matches metadata:  {TRAINING_TOKENIZER.decode_ids(row_input_ids.tolist(), strip_pad=True) == row_input_tokens}")
print(f"decoded target matches metadata: {TRAINING_TOKENIZER.decode_ids(row_target_ids.tolist(), strip_pad=True) == row_target_tokens}")

nonpad_target_count = int(batch["target_attention_mask"][row_index].sum())
print("\nTarget ids / labels around the padding boundary for row 0:")
boundary_rows = []
start = max(0, nonpad_target_count - 3)
end = min(row_target_ids.numel(), nonpad_target_count + 5)
for position in range(start, end):
    token = TRAINING_TOKENIZER.decode_ids([int(row_target_ids[position])])[0]
    boundary_rows.append(
        {
            "position": position,
            "token": token,
            "target_id": int(row_target_ids[position]),
            "label_id": int(row_labels[position]),
            "mask": int(batch["target_attention_mask"][row_index, position]),
        }
    )
print_table(boundary_rows, ["position", "token", "target_id", "label_id", "mask"])

assert batch["input_ids"].shape == (3, 256)
assert batch["target_ids"].shape == (3, 64)
assert batch["labels"].shape == (3, 64)
assert row_input_tokens[-1] == "<EDIT>"
assert row_target_tokens[0].startswith("<POS_")
assert row_target_tokens[-1] == TRAINING_TOKENIZER.eos_token


Batch tensor fields:
input_ids                shape=(3, 256) dtype=torch.int64 values=<matrix>
input_attention_mask     shape=(3, 256) dtype=torch.int64 values=<matrix>
target_ids               shape=(3, 64) dtype=torch.int64 values=<matrix>
target_attention_mask    shape=(3, 64) dtype=torch.int64 values=<matrix>
labels                   shape=(3, 64) dtype=torch.int64 values=<matrix>
num_mutations            shape=(3,) dtype=torch.int64 values=[1, 2, 1]
used_random_init         shape=(3,) dtype=torch.bool values=[False, False, False]
pair_index               shape=(3,) dtype=torch.int64 values=[0, 1, 2]
input_length             shape=(3,) dtype=torch.int64 values=[58, 40, 61]
target_length            shape=(3,) dtype=torch.int64 values=[6, 4, 4]

Metadata lists have one entry per batch row:
input_tokens                   len=3
target_tokens                  len=3
current_prefix                 len=3
target_integrand_prefix        len=3
target_antiderivative_prefix   len=3
warning_coun

## Closing Notes

- Prefix strings are the human-readable surface, but the mutation code works on typed AST nodes.
- `canonicalize(...)` defines the stable tree representation used for equality, node ids, token spans, and subtree sizes.
- `index_tree_positions(...)` assigns preorder ids and canonical token spans, and `sigma_small` limits mutable nodes by `subtree_size`.
- `local_const_edit` changes constant leaves directly; `local_same_arity_replacement` preserves local shape and children; `sampled_small_subtree_replacement` can replace a selected node with any supported small `Expr` subtree.
- `mutate_once(...)` is the forward noising engine and returns `MutationResult` metadata pointing back to the selected location in the pre-mutation canonical tree.
- `build_observation(...)` packages only inference-time state from `(f, I_t)`: the target integrand, current candidate, derivative, and optional residual features.
- Observation intentionally excludes reverse edit labels and the gold antiderivative `I*`; those stay separate from model input and belong to supervision/debug metadata.
- `first_edit_toward_target(...)` creates the next useful reverse repair edit from the visible current tree toward `I*`, including broad small-subtree jumps when they help.
- `TreeDiffusionTokenizer(...)` turns observations and edit targets into deterministic token sequences: field-wrapped observation plus `<EDIT>` for input, and `<POS_i> replacement_subtree <eos>` for the label.
- `generate_current_candidate(...)` mixes bounded mutation walks with bounded random initialization through `rho`; `smax` limits mutation-walk length.
- `generate_training_example(...)` computes labels with the reverse edit path, so supervision is not tied to the hidden final forward mutation.
- Encoded examples use tokenizer padding and round-trip decoding; dataloader batches add attention masks and `labels` with padding replaced by `-100`.
- Metadata fields such as `input_tokens`, `target_tokens`, and prefix strings are kept in batches for inspection; tensor fields are ready for model input and loss computation.
